<a href="https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/main/HW/2026/05-monitoring/hw-05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Homework: Monitoring

In module 5 we learned how to monitor our RAG system: capture metrics
from each LLM call, store them in a database, and visualize them on a
dashboard.

In the module we built all of this by hand - a custom dataclass for
the metrics, PostgreSQL for storage, Streamlit and Grafana for
dashboards.

In this homework, we will explore an alternative: [OpenTelemetry](https://opentelemetry.io/) (OTel).
This is the industry standard for code instrumentation. Every monitoring
framework we mentioned is built
on top of it - like Logfire, Langfuse, Arize Phoenix and others.

In this homework we will use OTel directly. We will instrument our
RAG with traces, capture metrics as span attributes, persist the
spans to SQLite, and build a dashboard from the trace data.

We keep using the same course-lessons RAG from homework 1. The
knowledge base is the 72 lesson pages pulled from GitHub, indexed
with minsearch.

> It's possible your answers won't match exactly. If so, select the closest one.

## Setup

Create a fresh project:


In [ ]:
%%bash
pip install gitsource minsearch groq python-dotenv

We want everyone to start with the same code, so we prepared a starter package.

Download it:

In [ ]:
%%bash
PREFIX=https://raw.githubusercontent.com/Jaguar838/llm-zoomcamp/main/HW/2026/05-monitoring
wget $PREFIX/rag_helper.py
wget $PREFIX/starter.py

We keep things simpler and focus only on RAG. However, all the concepts could be directly translated to agents.

Next, you need to put your OpenAI key in a `.env` file:

In [3]:
import os
from google.colab import userdata

# Retrieve GROQ_API_KEY from Colab secrets
groq_api_key = userdata.get('GROQ_API_KEY')

# Write both API keys to a .env file
with open('.env', 'w') as f:
    f.write(f'GROQ_API_KEY={groq_api_key}\n')

print("'.env' file created/updated with GROQ_API_KEY.")

'.env' file created/updated with GROQ_API_KEY.


Like previously, you can use any alternative you want.

The starter loads the 72 course lessons, builds a text-search index,
and wraps it in a `RAGBase` instance you can call right away:

In [4]:
from groq import Groq

from gitsource import GithubRepositoryDataReader
from minsearch import Index

from dotenv import load_dotenv
load_dotenv()

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = Groq()

For the LLM, we recommend OpenAI with `gpt-5.4-mini`, but you can use
any model and provider you want.

## OpenTelemetry setup

First, install the OpenTelemetry libraries:

In [ ]:
!pip install opentelemetry-api opentelemetry-sdk

- `opentelemetry-api` is the interface - the classes and functions you
import in your code (`trace`, `Tracer`, `Span`)
- `opentelemetry-sdk` is the implementation that actually creates and processes spans.

## OpenTelemetry

Before we start, we need to learn a few concepts from OTel - we will
use them in this homework.

- A trace is the end-to-end story of a single request as it moves
  through your system. For us, it's one RAG call.
- A span is one operation within a trace. A trace is made of one
  or more spans, organized as a tree. Each span has a name, a start
  and end time, and a set of attributes. For us we will have one span
  inside the trace, but for agents one trace will have multiple spans.
- Attributes are key-value pairs attached to a span - anything you
  want to record, like the number of tokens used or the cost of a call.

When a span finishes - meaning the code block it wraps completes - the
SDK hands it to a span processor, which forwards it to an exporter.
The exporter decides where the span goes: to the console, to a file,
to a database, or to a remote collector. We will see all of this in
practice in the questions below.

We start with the `ConsoleSpanExporter`, which prints each finished
span to the terminal so we can see what OTel captures:

In [6]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

class CapturingConsoleExporter(ConsoleSpanExporter):
    def __init__(self):
        super().__init__()
        self.spans = []

    def export(self, spans):
        self.spans.extend(spans)
        return super().export(spans)


console_exporter = CapturingConsoleExporter()

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Here is what each line does:

- `TracerProvider()` creates the SDK's central configuration object.
  It owns the span processors and decides how spans are built.
- `SimpleSpanProcessor(ConsoleSpanExporter())` wires a processor that
  forwards every finished span to the console exporter, one at a time.
  "Simple" means synchronous and immediate - good for development.
- `trace.set_tracer_provider(provider)` registers the provider
  globally, so every call to `trace.get_tracer(...)` returns a tracer
  backed by it.
- `trace.get_tracer("llm-zoomcamp")` returns a `Tracer` we use to
  create spans. The string is just a label for the instrumentation
  scope - it identifies which part of the code produced the spans.

Put this block at the top of your script, before you import or use
`starter` - so the tracer provider is ready before any code that
might create spans.

With the tracer in hand, you can wrap any block of code in a span:

In [7]:
# with tracer.start_as_current_span("my_operation") as span:
#     # your code here
#     span.set_attribute("my_key", "my_value")

`start_as_current_span` creates a new span and makes it the "current"
span for the duration of the `with` block. Any code inside the block -
including other calls to `start_as_current_span` - becomes a child of
this span. When the block exits, the span ends automatically.

You will use this pattern to instrument the RAG methods in the
questions below.

## Q1. First trace

Wrap the `rag()` method so each call produces a span. The simplest way
is to create a `RAGTraced` subclass of `RAGBase` that wraps `rag()`,
`search()`, and `llm()` each in their own span.

Run this query:

> How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary.
Count the spans in the console output - each one is a separate
`ReadableSpan` entry. How many spans does the trace produce?

* 1
* 3
* 5
* 7

In [14]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            results = super().search(query, num_results=num_results)
            span.set_attribute("num_results", len(results))
            return results

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            span.set_attribute("prompt", prompt)
            response = super().llm(prompt)

            if hasattr(response, 'usage') and response.usage:
                input_tokens = getattr(response.usage, 'prompt_tokens', 0)
                output_tokens = getattr(response.usage, 'completion_tokens', 0)
                span.set_attribute("input_tokens", input_tokens)
                span.set_attribute("output_tokens", output_tokens)
                cost = (input_tokens * 0.10 + output_tokens * 0.40) / 1_000_000
                span.set_attribute("cost", cost)
                print(f"\n>>> Token usage: input={input_tokens}, output={output_tokens}, cost=${cost:.6f}")

            return response.choices[0].message.content if hasattr(response, 'choices') else response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            answer = super().rag(query)
            span.set_attribute("answer", str(answer))
            return answer

assistant = RAGTraced(index=index, llm_client=client, model='openai/gpt-oss-20b')

In [9]:
query = "How does the agentic loop keep calling the model until it stops?"

In [15]:
with tracer.start_as_current_span("rag_call") as span:
    span.set_attribute("query", query)
    answer = assistant.rag(query)
    span.set_attribute("answer", str(answer))
    print(f"Answer: {answer}")

# Count spans captured in the current trace
print(f"Total spans captured: {len(console_exporter.spans)}")

{
    "name": "search",
    "context": {
        "trace_id": "0xaf222f097b64626ad343bcdad7e859aa",
        "span_id": "0xad41e401a90fc2e0",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x0b9c43163806a584",
    "start_time": "2026-07-22T15:04:10.145262Z",
    "end_time": "2026-07-22T15:04:10.148634Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.42.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xaf222f097b64626ad343bcdad7e859aa",
        "span_id": "0x0a0033694a884184",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x0b9c43163806a

In [18]:
# Q1: how many spans does the trace produce?
for s in console_exporter.spans:
    print(f"Q3 - span '{s.name}'")

Q3 - span 'search'
Q3 - span 'llm'
Q3 - span 'rag'
Q3 - span 'rag_call'
Q3 - span 'search'
Q3 - span 'llm'
Q3 - span 'rag'
Q3 - span 'rag_call'
Q3 - span 'search'
Q3 - span 'llm'
Q3 - span 'rag'
Q3 - span 'rag_call'


In [ ]:
Q1: 3

## Q2. Capturing metrics as span attributes

Spans are not just timing markers - you can attach any information you
want to them with `set_attribute`. We already use spans to record how
long each step takes. Now we'll add the metrics we care about: tokens
and cost.

Read the token usage from the LLM response (the `llm()` method in the
starter already returns the raw response object) and set them as
attributes on the `llm` span:


And since we know both input and output tokens, we can also compute
the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

* 700
* 7000
* 70000
* 700000

> These numbers vary between runs. Pick the closest option.

In [20]:
llm_span = next(s for s in console_exporter.spans if s.name == "llm")
input_tokens = dict(llm_span.attributes)["input_tokens"]
print("Q2 - llm span input_tokens:", input_tokens)

KeyError: 'input_tokens'

## Q3. Span timing

Each span automatically records its duration. Look at the console output
from Q1 and find the durations for the `search` span and the `llm` span.

For a typical query, roughly how long does the LLM call take?

* Under 100ms
* 100-500ms
* 500-2000ms
* Over 2000ms

> The first call can be slower (cold start). Pick the range you see
> most often.

In [17]:
# Q3: span durations (nanoseconds -> ms)
for s in console_exporter.spans:
    duration_ms = (s.end_time - s.start_time) / 1_000_000
    print(f"Q3 - span '{s.name}' duration_ms: {duration_ms:.1f}")

Q3 - span 'search' duration_ms: 6.9
Q3 - span 'llm' duration_ms: 2381.2
Q3 - span 'rag' duration_ms: 2393.7
Q3 - span 'rag_call' duration_ms: 2397.2
Q3 - span 'search' duration_ms: 4.6
Q3 - span 'llm' duration_ms: 167.4
Q3 - span 'rag' duration_ms: 175.9
Q3 - span 'rag_call' duration_ms: 179.0
Q3 - span 'search' duration_ms: 3.4
Q3 - span 'llm' duration_ms: 1635.5
Q3 - span 'rag' duration_ms: 1642.9
Q3 - span 'rag_call' duration_ms: 1645.0


In [19]:
print("Answer Q3: 500-2000ms")

Answer Q3: 500-2000ms


## Q4. Saving traces to SQLite

Right now the spans are printed to the terminal and then gone. We don't
save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than
Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there.
But we need to create a custom exporter. Instead of printing the spans,
it will save them to the database.

OTel calls the exporter through the same span processor we already use,
we just swap the destination.

Now we will create a custom exporter that saves each finished span to a
SQLite database. The exporter extends `SpanExporter`. It has the following methods:

- `export` method that receives a list of `ReadableSpan` objects
- `shutdown` and `force_flush` methods

Let's implement it:

In [21]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

Replace the console exporter with this new exporter:

In [22]:
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

Re-run the query from Q1. Which span names appear in the `spans` table?

* Only `rag`
* `rag` and `llm`
* `rag`, `search`, and `llm`
* `search`, `llm`, and `judge`

In [23]:
answer = assistant.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x1f4b26d41a4872cfa11e6bf95e6e20e3",
        "span_id": "0xe0c7705663175fe2",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xd6f0e94cf9e10ccc",
    "start_time": "2026-07-22T15:17:52.358221Z",
    "end_time": "2026-07-22T15:17:52.368589Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.42.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x1f4b26d41a4872cfa11e6bf95e6e20e3",
        "span_id": "0x1382016da2cd720e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xd6f0e94cf9e10

In [24]:
import pandas as pd

df = pd.read_sql("SELECT * FROM spans", sqlite3.connect("traces.db"))
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784733472358221312,1784733472368588873,None,None,None
1,llm,1784733472387139097,1784733474241487142,None,None,None
2,rag,1784733472358063831,1784733474252133554,None,None,None


In [25]:
# Q4: which span names appear in the spans table?
span_names = sorted(df["name"].unique().tolist())
print("Q4 - span names in traces.db:", span_names)

Q4 - span names in traces.db: ['llm', 'rag', 'search']


## Q5. Querying trace data

The traces are now in SQLite. Run one more query through the traced
RAG, then query the database.

The `rag` span wraps everything, so its duration includes both
`search` and `llm`. To see where time actually goes, exclude the
`rag` span and compare the children.

Using SQL (or pandas), compute the total duration for each span name
excluding `rag`. Which span type takes the most total time?

* `search`
* `llm`
* They're all about the same

In [26]:
# Q5: excluding rag, which span type takes the most total time?
df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1_000_000
totals = df[df["name"] != "rag"].groupby("name")["duration_ms"].sum()
print("Q5 - total duration by span name (excluding rag):")
print(totals)

Q5 - total duration by span name (excluding rag):
name
llm       1854.348045
search      10.367561
Name: duration_ms, dtype: float64


## Q6. Token stability across runs

Load the SQLite data with pandas. One thing a dashboard can tell you
is how stable your system is. If the same query always produces the
same number of input tokens, the context your RAG retrieves is
consistent. If it varies a lot, something in the search may be
unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls
total in the database). Then compute the input tokens for each `llm`
span.

How much do the input tokens vary across these 4 runs?

* They're identical
* Within 10% of each other
* Within 50% of each other
* They vary more than 50%

In [27]:
# Q6: run the same query 3 more times (4 total RAG calls in the database,
# counting the one just above), then check input-token stability.
for i in range(3):
    assistant.rag(query)

df = pd.read_sql("SELECT * FROM spans WHERE name = 'llm'", sqlite3.connect("traces.db"))
print("Q6 - llm rows in traces.db:", len(df))
print("Q6 - input_tokens per run:", df["input_tokens"].tolist())

mean_tokens = df["input_tokens"].mean()
max_dev = (df["input_tokens"] - mean_tokens).abs().max()
pct_variance = (max_dev / mean_tokens) * 100 if mean_tokens else 0
print(f"Q6 - max deviation from mean: {pct_variance:.2f}%")

{
    "name": "search",
    "context": {
        "trace_id": "0x2f7d3ab52ff169b9c3bf42c903bf2ec9",
        "span_id": "0x90df088970f6ac84",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x8400109e65bf03d8",
    "start_time": "2026-07-22T15:23:34.494754Z",
    "end_time": "2026-07-22T15:23:34.498742Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.42.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x2f7d3ab52ff169b9c3bf42c903bf2ec9",
        "span_id": "0x74cd2f03eabc4b06",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x8400109e65bf0

In [28]:
print("=== Answers summary ===")
print("Q1 (span count):        ", num_spans)
print("Q2 (llm input tokens):  ", input_tokens)
print("Q4 (span names):        ", span_names)
print("Q5 (duration by name):  ")
print(totals)
print("Q6 (llm input_tokens across 4 runs):", df["input_tokens"].tolist(), f"{pct_variance:.2f}%")

=== Answers summary ===
Q1 (span count):         12


NameError: name 'input_tokens' is not defined


## Going further

We built a custom SQLite exporter to understand how OTel works under
the hood. In practice you rarely instrument everything by hand.

### Collectors and backends

Instead of writing your own exporter, you
send spans to an
[OTel Collector](https://opentelemetry.io/docs/collector/), which
forwards them to a backend like
[Jaeger](https://www.jaegertracing.io/),
[Tempo](https://grafana.com/oss/tempo/), or a managed service. The
collector handles batching, retries, and routing so your app does not
have to. Jaeger (or Grafana's Tempo) then gives you a UI to browse
traces, filter by span name, and drill into timing - the same things
we did by querying SQLite, but interactive and built for scale.

### Auto-instrumentation

Most ecosystems have OTel wrappers that add
spans for you. For Python there is
`opentelemetry-instrumentation-openai` and similar libraries for
popular frameworks. You call one or two lines of setup and get LLM
spans, token counts, and tool calls traced automatically - no
subclassing, no manual `set_attribute`.

Frameworks like
[Pydantic Logfire](https://logfire.dev/) build on top of OTel and
take it even further: you get a hosted dashboard, automatic
instrumentation for Pydantic AI agents, and structured logging - all
with minimal code. We used Logfire in the
[dlt workshop homework](../workshops/dlt/homework.md), where we
instrumented an agent and pulled the traces back out with dlt. This
homework is the manual version of the same idea: same OTel standard
underneath, just more hands-on.